# 02 — Scorers, calibration and evaluation

Reads the cache written by `01_retrieval_and_generation.ipynb` and produces every table and figure of the paper.
A CPU runtime is enough.

| Step | Output |
|---|---|
| 1. Inventory | `results/tables/inventory.csv` |
| 2. Scorers | `results/scores/*.csv.gz`, `results/tables/auroc.csv` |
| 3. Evaluation (10 methods × 6 α × 300 splits) | `results/tables/results_all.csv` |
| 4. Sensitivity, action shares | `results/tables/sensitivity.csv`, `action_shares.csv` |
| 5. Distribution shift | `results/tables/shift.csv` |
| 6. Compute cost | `results/tables/compute_cost.csv` |
| 7. Figures and summary | `figures/`, numbers quoted in the text |

Steps 3, 4 and 7 only need `results/scores`, which is part of the repository, so they can be rerun without the cache.
Steps 1, 2, 5 and 6 need the cache (hidden states are not distributed because of their size).

## 0. Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/risk-controlled-adaptive-rag"):
        !git clone -q https://github.com/erturkerdagi/risk-controlled-adaptive-rag.git /content/risk-controlled-adaptive-rag
    %cd /content/risk-controlled-adaptive-rag
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/rarag"   # same folder as in notebook 01
CACHE = f"{DRIVE_ROOT}/cache"
RUN_TAG = "full"
HAVE_CACHE = os.path.isdir(CACHE)
print("cache found" if HAVE_CACHE else "no cache: steps 1, 2, 5 and 6 are skipped")

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rarag import config as C
from rarag.data import available_pairs, load_pair
from rarag.scorers import Scorer, safe_auc, train_pool_split
from rarag.shift import shift_experiment

warnings.filterwarnings("ignore", category=FutureWarning)
os.makedirs("results/scores", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
PAIRS = available_pairs(CACHE, C.MODELS, C.DATASETS, RUN_TAG) if HAVE_CACHE else []
print(len(PAIRS), "model-dataset pairs")

## 1. Inventory

Labels are recomputed from the stored answers with the containment rule, so every pair uses the same label.

In [ ]:
if PAIRS:
    inv = []
    for m, d in tqdm(PAIRS):
        df, _, _, layers = load_pair(CACHE, m, d, RUN_TAG)
        y0, y1 = df.y0.values, df.y1.values
        inv.append({"model": m, "dataset": d, "n": len(df), "acc_closed_book": y0.mean(), "acc_rag": y1.mean(),
                    "oracle": np.maximum(y0, y1).mean(), "retrieval_helps": ((y0 == 0) & (y1 == 1)).mean(),
                    "retrieval_hurts": ((y0 == 1) & (y1 == 0)).mean(), "hidden_layers": list(map(int, layers))})
    pd.DataFrame(inv).to_csv("results/tables/inventory.csv", index=False)
pd.read_csv("results/tables/inventory.csv").round(3)

## 2. Scorers

Each pair is split once (seed 2026): one half trains the scorers, the other half is the pool used for calibration
and testing. The layer, the PCA projection, the regularisation strength and whether the hidden-state block is kept
are all chosen on the training half. The pool keeps only what the evaluation needs.

In [ ]:
KEEP = ["id", "y0", "y1", "ent_mean_0", "ent_mean_1", "margin_first_0"]
auc_rows = []
for m, d in tqdm(PAIRS):
    df, H0, H1, layers = load_pair(CACHE, m, d, RUN_TAG)
    tr, te = train_pool_split(len(df), C.TRAIN_FRAC, C.SEED)
    sc = Scorer(seed=C.SEED).fit(df, H0, H1, tr)
    pool = df.iloc[te].reset_index(drop=True)[KEEP].copy()
    pool[["y0", "y1"]] = pool[["y0", "y1"]].astype(int)
    for k, v in sc.scores(df.iloc[te], H0[te], H1[te]).items():
        pool[k] = v
    pool.to_csv(f"results/scores/{m}__{d}.csv.gz", index=False, float_format="%.17g")
    y0, y1 = pool.y0, pool.y1
    auc_rows.append({
        "model": m, "dataset": d, "layer_q": int(layers[sc.l0]), "layer_rag": int(layers[sc.l1]),
        "hidden_in_s0": "h0" in sc.choice["s0"], "hidden_in_s1": "h1" in sc.choice["s1"],
        "y0: -entropy": safe_auc(y0, -pool.ent_mean_0), "y0: hidden probe": safe_auc(y0, pool.s0_probe),
        "y0: Proposed (no hidden states)": safe_auc(y0, pool.s0_nohidden), "y0: Proposed": safe_auc(y0, pool.s0),
        "y1: -entropy": safe_auc(y1, -pool.ent_mean_1),
        "y1: Proposed (no hidden states)": safe_auc(y1, pool.s1_nohidden), "y1: Proposed": safe_auc(y1, pool.s1)})
if auc_rows:
    pd.DataFrame(auc_rows).to_csv("results/tables/auroc.csv", index=False)
pd.read_csv("results/tables/auroc.csv").round(3)

## 3. Evaluation

All ten methods go through the same calibration; they differ only in the scores. Each pool is split 300 times into
equal calibration and test halves, for every α. About ten minutes on a CPU.

In [ ]:
!python scripts/evaluate.py
res = pd.read_csv("results/tables/results_all.csv")
res[res.alpha == C.ALPHA_MAIN].pivot_table(index=["model", "dataset"], columns="method",
                                           values="coverage").round(3)

## 4. Sensitivity and action shares

In [ ]:
!python scripts/sensitivity.py
!python scripts/action_shares.py

## 5. Distribution shift

For Qwen2.5-7B and Llama-3.1-8B: scorers and thresholds from a source dataset applied to every other dataset
(m = 0), and the same scorers with thresholds recalibrated on m labelled target questions.

In [ ]:
if PAIRS:
    rows = []
    for model in ["Qwen2.5-7B-Instruct", "Llama-3.1-8B-Instruct"]:
        ds = [d for d in C.DATASETS if (model, d) in PAIRS]
        rows += shift_experiment(CACHE, model, ds, run_tag=RUN_TAG, progress=tqdm)
    pd.DataFrame(rows).to_csv("results/tables/shift.csv", index=False)
shift = pd.read_csv("results/tables/shift.csv")
shift.groupby(["model", "m"])[["violation", "coverage"]].agg(["mean", "max"]).round(3)

## 6. Compute cost (from the log of notebook 01)

In [ ]:
status_file = f"{DRIVE_ROOT}/logs/run_status_{RUN_TAG}.json"
if os.path.exists(status_file):
    st = json.load(open(status_file))
    cost = pd.DataFrame([{"model": k.split("/")[0], "dataset": k.split("/")[1], **v}
                         for k, v in st.items() if "_load_error" not in k])
    cost["gpu_hours"] = cost.n * cost.sec_per_q / 3600
    cost.to_csv("results/tables/compute_cost.csv", index=False)
cost = pd.read_csv("results/tables/compute_cost.csv")
cost.groupby("model").agg(datasets=("dataset", "size"), questions=("n", "sum"),
                          gpu_hours=("gpu_hours", "sum")).round(2)

## 7. Figures and summary

In [ ]:
!python scripts/make_figures.py
!python scripts/summarize.py